In [ ]:
import yaml 

# Load the credentials YAML file
credential_path = '../credentials.yml' 
cfg = yaml.load(open(credential_path, "r"), Loader=yaml.Loader)
cfg

In [ ]:
from sqlalchemy import create_engine

# Database credentials
user = cfg['postgresql']['username']
password = cfg['postgresql']['password']
host = 'cm-de-k1-db.c9ms60q6cw37.ap-southeast-2.rds.amazonaws.com'
port = '5432'
database = 'postgres'

# Construct the connection string
# The 'postgresql+psycopg2' part specifies the dialect and driver
connection_string = f'postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}'

# Create the SQLAlchemy engine
engine = create_engine(connection_string)
connection = engine.connect()

Read data 

In [ ]:
# HINTS
# 1. Read data from SQL table coffee.sales
# 2. Read all data and export to 
import pandas as pd

schema = 'coffee'
table = 'sales'

df_postgres = pd.read_sql_table(table_name=table, con=connection, schema=schema)
df_postgres.tail(5)

Load to storage

In [ ]:
import sqlite3

# create destination database
dest_database_file = "destination.db"
dest_conn = sqlite3.connect(dest_database_file)

# 1. Upload data to a table named: coffee_sales
sqlite_table_name = 'coffee_sales'
df_postgres.to_sql(
    name=sqlite_table_name,
    con=dest_conn,
)

In [43]:
# Read data from 'coffee_sales' table
db_connection_string = 'sqlite:///destination.db'
db_engine = create_engine(url=db_connection_string)
db_conn = db_engine.connect()

df_coffee_sales = pd.read_sql_table(table_name=sqlite_table_name, con=db_conn)
df_coffee_sales.tail(5)

,index,date,datetime,cash_type,card,money,coffee_name
5381,2688,2024-12-31,2024-12-31 17:10:17.537,card,ANON-0000-0000-0525,35.76,Latte
5382,2689,2024-12-31,2024-12-31 17:30:52.785,card,ANON-0000-0000-1064,30.86,Americano with Milk
5383,2690,2024-12-31,2024-12-31 17:31:48.543,card,ANON-0000-0000-1065,35.76,Cappuccino
5384,2691,2024-12-31,2024-12-31 19:07:25.413,card,ANON-0000-0000-1066,35.76,Latte
5385,2692,2024-12-31,2024-12-31 19:08:38.899,card,ANON-0000-0000-1066,35.76,Hot Chocolate


Incremental Loading

In [ ]:
# 1. Read latest data from storage
# 2. Filter incremental records from source
# 3. Load as 'append' to existing table

In [44]:
# Get the last loaded timestamp from destination.db
latest_dt = pd.read_sql('SELECT MAX(datetime) AS latest_datetime FROM coffee_sales', con=db_conn)
latest_dt = latest_dt['latest_datetime'].iloc[0]
# latest_dt = pd.to_datetime(latest_dt)
latest_dt

'2024-12-31 19:08:38.899000'

In [45]:
# Read the latest data from postgres that is later than latest_dt
df_latest = pd.read_sql_table(table_name=table, con=connection, schema=schema)
df_incremental = df_latest[df_latest["datetime"] > latest_dt]
df_incremental.tail(5)


,date,datetime,cash_type,card,money,coffee_name


In [47]:
# Load as 'append' to the existing sqlite table
df_incremental.to_sql(
    name=sqlite_table_name,
    con=dest_conn,
    if_exists='append'
)

0